# Assignment 4 - Reject Option based Classification (ROC)
Post-processing bias mitigation from Kamiran, Karim and Zhang (2012), *Decision Theory for Discrimination-aware Classification*.

Pipeline per dataset: load data -> fairness metrics on original labels -> stratified split (60/20/20 train/validation/test) -> logistic regression -> ROC with theta chosen on the validation set -> metrics on the test set.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

                  
SEED = 42
THETAS = np.round(np.arange(0.50, 0.951, 0.01), 2)   # theta grid used in the paper (0.5 to 0.95)
DP_TARGET = 0.05                      # accept a validation |DP_Diff| of at most 5 percentage points

## Fairness metrics


In [2]:
# Demographic Parity Difference
def dp_diff(y, s):
    return y[s == 1].mean() - y[s == 0].mean()

# Disparate Impact Ratio
def dir_ratio(y, s):
    return y[s == 0].mean() / y[s == 1].mean()

# Consistency
def consistency(X, y, k=5):
    idx = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X, return_distance=False)[:, 1:]
    return 1 - np.abs(y - y[idx].mean(axis=1)).mean()

## Loading the datasets
Each loader returns the features `X` (the sensitive attribute is removed), the label `y` (1 = favorable) and the group `s` (1 = favored).

In [3]:
def load_adult():
    d = pd.read_csv("adult.csv").replace("?", np.nan).dropna()
    y = (d["income"] == ">50K").astype(int).values       # favorable: income > 50K
    s = (d["sex"] == "Male").astype(int).values          # deprived: female
    X = d.drop(columns=["income", "sex", "fnlwgt", "education"])   # education duplicates education.num
    return X, y, s

def load_german():
    cols = ["status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
            "installment_rate", "personal_status", "other_debtors", "residence_since", "property",
            "age", "other_installments", "housing", "existing_credits", "job", "num_dependents",
            "telephone", "foreign_worker", "class"]
    d = pd.read_csv("german.data", sep=" ", header=None, names=cols)
    y = (d["class"] == 1).astype(int).values              # favorable: good credit
    s = (d["personal_status"] != "A92").astype(int).values   # deprived: A92 (female)
    X = d.drop(columns=["class", "personal_status"])
    return X, y, s

def load_compas():
    d = pd.read_csv("compas-scores-two-years.csv")
    d = d[(d.days_b_screening_arrest <= 30) & (d.days_b_screening_arrest >= -30)   # ProPublica filter
          & (d.is_recid != -1) & (d.c_charge_degree != "O") & (d.score_text != "N/A")]
    d = d[d.race.isin(["African-American", "Caucasian"])]
    y = (d["two_year_recid"] == 0).astype(int).values     # favorable: no recidivism
    s = (d["race"] == "Caucasian").astype(int).values     # deprived: African-American
    X = d[["sex", "age_cat", "priors_count", "c_charge_degree"]]   # no race, no COMPAS scores
    return X, y, s

LOADERS = {"Adult": load_adult, "German": load_german, "COMPAS": load_compas}

## ROC and the full pipeline


In [4]:
def roc_predict(p, s, theta):
    pred = (p >= 0.5).astype(int)
    critical = np.maximum(p, 1 - p) < theta
    pred[critical & (s == 0)] = 1
    pred[critical & (s == 1)] = 0
    return pred

def choose_theta(p_val, y_val, s_val):
    rows = []
    for t in THETAS:
        pred = roc_predict(p_val, s_val, t)
        rows.append((t, (pred == y_val).mean(), abs(dp_diff(pred, s_val))))
    ok = [r for r in rows if r[2] <= DP_TARGET]
    return max(ok, key=lambda r: r[1])[0] if ok else min(rows, key=lambda r: r[2])[0]

def evaluate(pred, y, s, X):
    return {"Accuracy": (pred == y).mean(), "DP_Diff": dp_diff(pred, s),
            "DIR": dir_ratio(pred, s), "Consistency": consistency(X, pred)}

def run(name):
    X, y, s = LOADERS[name]()
    X = pd.get_dummies(X).astype(float).values
    results = {}

    # metrics on the original labels (whole dataset, before any classifier or mitigation)
    Xall = StandardScaler().fit_transform(X)
    results["Original labels"] = {"Accuracy": np.nan, "DP_Diff": dp_diff(y, s),
                                  "DIR": dir_ratio(y, s), "Consistency": consistency(Xall, y)}

    # stratified 60/20/20 split (stratified on label and group together)
    strat = y * 2 + s
    X_a, X_te, y_a, y_te, s_a, s_te = train_test_split(X, y, s, test_size=0.2, stratify=strat, random_state=SEED)
    X_tr, X_val, y_tr, y_val, s_tr, s_val = train_test_split(
        X_a, y_a, s_a, test_size=0.25, stratify=y_a * 2 + s_a, random_state=SEED)

    scaler = StandardScaler().fit(X_tr)
    X_tr, X_val, X_te = scaler.transform(X_tr), scaler.transform(X_val), scaler.transform(X_te)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    p_val, p_te = clf.predict_proba(X_val)[:, 1], clf.predict_proba(X_te)[:, 1]

    theta = choose_theta(p_val, y_val, s_val)
    results["Classifier, no mitigation"] = evaluate(roc_predict(p_te, s_te, 0.5), y_te, s_te, X_te)
    results["Classifier + ROC"] = evaluate(roc_predict(p_te, s_te, theta), y_te, s_te, X_te)
    print(f"{name}: n={len(y)}, test n={len(y_te)}, favorable rate={y.mean():.3f}, "
          f"deprived share={(s == 0).mean():.3f}, chosen theta={theta}")
    return pd.DataFrame(results).T.round(3), theta

## Run on the three datasets

In [5]:
all_results = {}
for name in LOADERS:
    table, theta = run(name)
    all_results[name] = table
    display(table) if "display" in globals() else print(table)
    print()

Adult: n=30162, test n=6033, favorable rate=0.249, deprived share=0.324, chosen theta=0.65
                           Accuracy  DP_Diff    DIR  Consistency
Original labels                 NaN    0.200  0.362        0.794
Classifier, no mitigation     0.848    0.178  0.309        0.920
Classifier + ROC              0.835    0.060  0.659        0.922

German: n=1000, test n=200, favorable rate=0.700, deprived share=0.310, chosen theta=0.52
                           Accuracy  DP_Diff    DIR  Consistency
Original labels                 NaN    0.075  0.897        0.669
Classifier, no mitigation     0.760   -0.024  1.030        0.770
Classifier + ROC              0.755   -0.046  1.060        0.755

COMPAS: n=5278, test n=1056, favorable rate=0.530, deprived share=0.602, chosen theta=0.55
                           Accuracy  DP_Diff    DIR  Consistency
Original labels                 NaN    0.132  0.783        0.585
Classifier, no mitigation     0.651    0.210  0.711        0.988
Classifier 